# 05 — Biological validation: cell composition & batch-effect diagnostics

This notebook (1) estimates **whole-blood leukocyte proportions** from Illumina 450K betas using a **Houseman-style constrained projection** against the **Reinius** reference (`450K_reinius_12_reference.csv` from Biolearn), and correlates **`age_acceleration`** with inferred fractions.

(2) **Batch-effect diagnostics:** combine **GSE40279** (training cohort) and **GSE87571** (external validation cohort) on the **same model CpGs**, run **PCA**, optional **UMAP**, distribution comparisons, and a **cohort classifier** to probe whether methylation space separates studies (technical structure) independently of age.

**Prerequisites:** `01_data_loading.ipynb`, `02_model_training.ipynb`. For external methylation, **`04_external_validation.ipynb`** should have been run once so `data/raw/GSE87571/` matrices exist—or this notebook downloads them the same way.

**No ComBat or harmonization** is applied here; this layer is **diagnostic only**.


## Dependencies

In [ ]:
%pip install -q pandas numpy scipy scikit-learn matplotlib joblib pyarrow seaborn GEOparse


## Paths (`pathlib` only; notebook cwd = `notebooks/`)

In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
PUBLIC_DIR = RESULTS_DIR / "public"
REF_DIR = DATA_DIR / "reference"

for d in (PROCESSED_DIR, RESULTS_DIR, PUBLIC_DIR, REF_DIR):
    d.mkdir(parents=True, exist_ok=True)

REFERENCE_URL = (
    "https://raw.githubusercontent.com/bio-learn/biolearn/master/"
    "biolearn/data/450K_reinius_12_reference.csv"
)
REFERENCE_PATH = REF_DIR / "450K_reinius_12_reference.csv"

RANDOM_STATE = 42
N_TOP_CPGS = 5000
TARGET_CPG = "cg16867657"
STRONG_CORR_THRESHOLD = 0.25

## Why cell composition matters (confounding)

Whole blood is a mixture of leukocyte subsets whose **DNA methylation profiles differ systematically**. Age-associated shifts in **immune cell proportions** (e.g., naive lymphocyte decline, myeloid bias) can produce **aggregated beta changes** that supervised age models absorb as “aging” signal. **Epigenetic age acceleration** — residuals from a clock — may therefore partly reflect **composition change** rather than cell-intrinsic aging programs.

Leukocyte shifts can **mimic aging** because many Horvath/Hannum clock probes were identified in blood and co-vary with immune turnover. **Strong** correlations between acceleration and deconvolved fractions suggest **substantial composition confounding**; **weak** correlations support interpreting residuals as more clock-specific (still not proof of causality).

## Load cohort methylation, ages, and frozen model

In [ ]:
import joblib
import numpy as np
import pandas as pd

x_path = PROCESSED_DIR / "X.pkl.gz"
y_path = PROCESSED_DIR / "y.csv"
scaler_path = PROCESSED_DIR / "scaler.joblib"
model_path = PROCESSED_DIR / "elasticnet_model.joblib"
selected_path = PROCESSED_DIR / "selected_cpgs.csv"

for p in (x_path, y_path, scaler_path, model_path, selected_path):
    if not p.is_file():
        raise FileNotFoundError("Run notebooks 01 and 02 first. Missing: " + str(p))

X = pd.read_pickle(x_path, compression="gzip").astype(np.float32)
y_age = pd.read_csv(y_path, index_col=0)["age"].astype(float)
common = X.index.intersection(y_age.index).sort_values()
X = X.loc[common]
y_age = y_age.loc[common]

scaler = joblib.load(scaler_path)
model = joblib.load(model_path)
selected_cpgs = pd.read_csv(selected_path)["cpg"].astype(str).tolist()

Z_all = scaler.transform(X[selected_cpgs])
pred_age = pd.Series(model.predict(Z_all), index=X.index, name="predicted_age")
age_accel = pred_age - y_age
age_accel.name = "age_acceleration"

meta = pd.DataFrame({"chronological_age": y_age, "predicted_age": pred_age, "age_acceleration": age_accel})
print("Samples:", meta.shape[0])
display(meta.head())

## Download reference matrix (Reinius / Biolearn)

In [ ]:
import urllib.request

if not REFERENCE_PATH.is_file():
    print("Downloading reference …")
    urllib.request.urlretrieve(REFERENCE_URL, REFERENCE_PATH)
else:
    print("Using cached reference:", REFERENCE_PATH)

ref = pd.read_csv(REFERENCE_PATH, index_col=0)
ref.index = ref.index.astype(str)
cell_types = list(ref.columns)
print("Cell-type columns:", cell_types)
print("Reference probes:", ref.shape[0])

## Houseman-style projection (per sample)

For each sample vector **b** of beta values on overlapping probes, solve

\(\min_{w \ge 0,\, \mathbf{1}^\top w = 1} \| R w - b \|_2\)

with **`scipy.optimize.lsq_linear`** (`R` = reference probe × cell-type matrix). This is the standard **non-negative sum-to-one** mixture fit used in reference-based blood DNAm deconvolution (Houseman et al., 2012).

In [ ]:
from scipy.optimize import minimize


def deconvolve_sample(beta_vec: np.ndarray, R: np.ndarray) -> np.ndarray:
    n_types = R.shape[1]

    def objective(w):
        residual = R @ w - beta_vec
        return float(np.dot(residual, residual))

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
    ]

    bounds = [(0.0, 1.0)] * n_types
    x0 = np.full(n_types, 1.0 / n_types)

    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-10},
    )

    if not res.success:
        raise RuntimeError(res.message)

    return res.x


probe_overlap = ref.index.intersection(X.columns.astype(str))
probe_overlap = probe_overlap.sort_values()

R_mat = ref.loc[probe_overlap].to_numpy(dtype=np.float64)
X_sub = X[probe_overlap].to_numpy(dtype=np.float64)

print("Probes used in deconvolution:", len(probe_overlap))

n_samp = X_sub.shape[0]
frac = np.zeros((n_samp, len(cell_types)), dtype=np.float64)

failed = 0

for i in range(n_samp):
    b = X_sub[i, :]

    mask = (
        np.isfinite(b)
        & np.all(np.isfinite(R_mat), axis=1)
    )

    if mask.sum() < max(50, R_mat.shape[1] + 2):
        frac[i, :] = np.nan
        failed += 1
        continue

    Ri = R_mat[mask, :]
    bi = b[mask]

    try:
        frac[i, :] = deconvolve_sample(bi, Ri)
    except Exception:
        frac[i, :] = np.nan
        failed += 1

comp = pd.DataFrame(frac, columns=cell_types, index=X.index)

print("Failed samples:", failed)
print("Valid samples:", comp.dropna().shape[0])
display(comp.head())
display(comp.describe())

## Merge with age acceleration and summarize

**Definition:** `age_acceleration = predicted_age − chronological_age` using the **same frozen model** as notebook 02 (no retraining).

In [ ]:
analysis = meta.join(comp)
analysis = analysis.dropna(subset=["age_acceleration"] + cell_types, how="any")
print("Samples with finite composition + acceleration:", analysis.shape[0])

## Correlations (Pearson)

In [ ]:
from scipy import stats

rows = []
for ct in cell_types:
    r, p = stats.pearsonr(analysis["age_acceleration"], analysis[ct])
    rows.append(
        {"cell_type": ct, "pearson_r": float(r), "p_value": float(p), "n": int(analysis.shape[0])}
    )

corr_df = pd.DataFrame(rows).sort_values("pearson_r", key=abs, ascending=False)
display(corr_df)

corr_path = RESULTS_DIR / "age_acceleration_celltype_correlations.csv"
corr_df.to_csv(corr_path, index=False)
shutil.copy2(corr_path, PUBLIC_DIR / corr_path.name)

## Correlation heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

mat = corr_df.set_index("cell_type")["pearson_r"].to_frame().T
mat.index = ["age_acceleration"]

fig, ax = plt.subplots(figsize=(10, 2.2))
sns.heatmap(mat, annot=True, fmt=".3f", cmap="RdBu_r", center=0.0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Pearson r: age acceleration vs cell proportion")
fig.tight_layout()
heat_path = RESULTS_DIR / "celltype_correlation_heatmap.png"
fig.savefig(heat_path, dpi=150)
plt.show()
shutil.copy2(heat_path, PUBLIC_DIR / heat_path.name)

## Scatter plots (one cell type per panel)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.ravel()
for ax, ct in zip(axes, cell_types):
    x = analysis[ct].to_numpy()
    y = analysis["age_acceleration"].to_numpy()
    ax.scatter(x, y, alpha=0.5, s=18)
    r, p = stats.pearsonr(x, y)
    ax.set_xlabel(ct)
    ax.set_ylabel("Age acceleration (y)")
    ax.set_title(f"r={r:.3f}, p={p:.2e}")
for ax in axes[len(cell_types) :]:
    ax.axis("off")
fig.suptitle("Age acceleration vs inferred leukocyte fraction")
fig.tight_layout()
sc_path = RESULTS_DIR / "age_accel_vs_celltype_scatters.png"
fig.savefig(sc_path, dpi=150)
plt.show()
shutil.copy2(sc_path, PUBLIC_DIR / sc_path.name)

## Export composition table

In [ ]:
comp_out = analysis[cell_types + ["chronological_age", "predicted_age", "age_acceleration"]].copy()
comp_export = comp.join(meta)
comp_export_path = RESULTS_DIR / "cell_composition_estimates.csv"
comp_export.to_csv(comp_export_path, index_label="sample_id_beta")
shutil.copy2(comp_export_path, PUBLIC_DIR / comp_export_path.name)
print("Wrote", comp_export_path.resolve())

## Interpretation (qualitative)

Inspect **`age_acceleration_celltype_correlations.csv`**. Large |r| with low *p* for neutrophil / lymphocyte axes often indicates that **residual epigenetic age** tracks **myeloid vs lymphoid balance**. Clocks trained without explicit adjustment can partly encode **immune skew** as “aging.” If correlations are uniformly near zero, composition is **unlikely to dominate** the acceleration signal in this cohort — though regional batch effects or non-leukocyte biology could still matter.

The optional section below fits an **exploratory** elastic net that **adds cell fractions as covariates** when correlations suggest worth investigating (threshold configurable above).

## Optional: covariate-adjusted Elastic Net (same cohort split)

Runs **only if** \(\max_k |r_k|\) between acceleration and cell fractions exceeds **`STRONG_CORR_THRESHOLD`**. This is **not** the primary frozen clock; it probes whether **aging prediction** changes when leukocyte fractions are explicit linear terms beside correlation-selected CpGs.

Pipeline mirrors notebook 02: **train/test split first**, **CpG screening on train only**, **scaler on selected CpGs only**, append **train-standardized cell proportions**, then **`ElasticNetCV`** on the training design.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

max_abs_r = float(corr_df["pearson_r"].abs().max())
print("max |Pearson r| (accel vs cell types):", round(max_abs_r, 4))

run_adjusted = max_abs_r >= STRONG_CORR_THRESHOLD
adj_metrics_path = RESULTS_DIR / "cell_adjusted_model_metrics.csv"
coef_compare_path = RESULTS_DIR / "cell_adjusted_coef_elovl2.csv"

if not run_adjusted:
    print(
        "Skipping adjusted model (|r| below threshold). "
        "Lower STRONG_CORR_THRESHOLD in the paths cell to force."
    )
else:
    X_train, X_test, y_train, y_test, comp_train, comp_test = train_test_split(
        X,
        y_age,
        comp,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )

    train_corr = X_train.corrwith(y_train)
    sel = train_corr.abs().sort_values(ascending=False).head(N_TOP_CPGS).index

    scaler_x = StandardScaler()
    X_tr_z = scaler_x.fit_transform(X_train[sel])
    X_te_z = scaler_x.transform(X_test[sel])

    scaler_c = StandardScaler()
    C_tr = scaler_c.fit_transform(comp_train[cell_types])
    C_te = scaler_c.transform(comp_test[cell_types])

    Xdesign_tr = np.hstack([X_tr_z, C_tr])
    Xdesign_te = np.hstack([X_te_z, C_te])

    names = list(sel.astype(str)) + [f"cell::{c}" for c in cell_types]

    en = ElasticNetCV(
        l1_ratio=[0.7, 0.8, 0.9, 0.95, 1.0],
        alphas=np.logspace(-3, 1, 30),
        cv=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_iter=30000,
    )
    en.fit(Xdesign_tr, y_train)
    pred_te = en.predict(Xdesign_te)

    mae_adj = mean_absolute_error(y_test, pred_te)
    rmse_adj = float(np.sqrt(mean_squared_error(y_test, pred_te)))
    r2_adj = r2_score(y_test, pred_te)
    pearson_adj = float(np.corrcoef(y_test, pred_te)[0, 1])

    coef_series = pd.Series(en.coef_, index=names)

    baseline_metrics = pd.read_csv(RESULTS_DIR / "model_metrics.csv")

    adj_row = pd.DataFrame(
        [
            {
                "model": "elasticnet_with_cell_covariates",
                "MAE": mae_adj,
                "RMSE": rmse_adj,
                "R2": r2_adj,
                "Pearson_r": pearson_adj,
                "alpha": float(en.alpha_),
                "l1_ratio": float(en.l1_ratio_),
                "n_nonzero_coef": int((en.coef_ != 0).sum()),
                "max_abs_accel_cell_r": max_abs_r,
            }
        ]
    )
    adj_row.to_csv(adj_metrics_path, index=False)
    shutil.copy2(adj_metrics_path, PUBLIC_DIR / adj_metrics_path.name)

    base_mae = float(baseline_metrics["MAE"].iloc[0])
    base_r2 = float(baseline_metrics["R2"].iloc[0])

    coef_elovl2_baseline = (
        float(model.coef_[selected_cpgs.index(TARGET_CPG)])
        if TARGET_CPG in selected_cpgs
        else np.nan
    )
    coef_elovl2_adj = (
        float(coef_series.loc[TARGET_CPG])
        if TARGET_CPG in coef_series.index
        else np.nan
    )

    pd.DataFrame(
        [
            {
                "cpg": TARGET_CPG,
                "coef_baseline_elasticnet": coef_elovl2_baseline,
                "coef_cell_adjusted_elasticnet": coef_elovl2_adj,
            }
        ]
    ).to_csv(coef_compare_path, index=False)
    shutil.copy2(coef_compare_path, PUBLIC_DIR / coef_compare_path.name)

    print("Baseline MAE / R2 (notebook 02):", base_mae, base_r2)
    print("Adjusted   MAE / R2:", mae_adj, r2_adj)
    display(adj_row)

## Batch-effect diagnostics (GSE40279 vs GSE87571)

External validation compares clocks across **different laboratories, preprocessing choices, and population composition**. DNA methylation is highly sensitive to **batch** and **biological** drivers; strong **cohort separability** in probe space suggests that apparent cross-study agreement could partly reflect **shared technical structure** rather than pure aging biology.

This section builds a **combined matrix** restricted to **`selected_cpgs`** that exist in both cohorts, then performs **PCA**, optional **UMAP**, age/cohort distribution checks, and a simple **cohort classifier**. High classifier performance indicates easy **study discrimination**—a flag for batch dominance, not proof that the clock is invalid.


### Why batch effects matter

- **Illumina arrays** integrate dye bias, bisulfite conversion, and normalization choices that differ by study.
- **Whole blood** mixes cell types; cohorts differ in age range and ancestry, which interact with technical factors.
- **Biological signal** (age-related CpG drift) can be **entangled** with **technical cohort structure** when validation uses public supplementary betas processed differently from training.

**Difference:** *Biological aging signal* is covariance with chronological age within a harmonized design; *technical cohort structure* is systematic divergence between studies that may align with age covariates by accident.


### Load external cohort (same pattern as `04_external_validation.ipynb`)


In [ ]:
import re
import urllib.request

import GEOparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RAW_DIR = DATA_DIR / "raw"
EXT_SUBDIR = RAW_DIR / "GSE87571"
EXT_SUBDIR.mkdir(parents=True, exist_ok=True)
EXT_GSE = "GSE87571"

BASE_SUPPL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE87nnn/GSE87571/suppl/"
downloads = {
    "GSE87571_matrix1of2.txt.gz": BASE_SUPPL + "GSE87571_matrix1of2.txt.gz",
    "GSE87571_matrix2of2.txt.gz": BASE_SUPPL + "GSE87571_matrix2of2.txt.gz",
}
for fname, url in downloads.items():
    dest = EXT_SUBDIR / fname
    if not dest.is_file():
        print("Downloading", fname, "…")
        urllib.request.urlretrieve(url, dest)
    else:
        print("Using existing", fname)


def read_probe_by_sample_matrix(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", compression="gzip", low_memory=False)
    probe_col = df.columns[0]
    df = df.rename(columns={probe_col: "probe_id"}).set_index("probe_id")
    df.columns = df.columns.astype(str)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


m1 = read_probe_by_sample_matrix(EXT_SUBDIR / "GSE87571_matrix1of2.txt.gz")
m2 = read_probe_by_sample_matrix(EXT_SUBDIR / "GSE87571_matrix2of2.txt.gz")

m1.columns = m1.columns.astype(str).str.strip()
m2.columns = m2.columns.astype(str).str.strip()

print("m1 shape probes × samples:", m1.shape)
print("m2 shape probes × samples:", m2.shape)

common_probes = m1.index.intersection(m2.index)

print("m1 probes:", m1.shape[0])
print("m2 probes:", m2.shape[0])
print("common probes:", len(common_probes))

if len(common_probes) < 100_000:
    raise ValueError(
        "Too few common probes between matrix halves. "
        "Inspect GSE87571 matrix files manually."
    )

m1 = m1.loc[common_probes]
m2 = m2.loc[common_probes]

# The files are split by samples, so concatenate columns, not rows.
beta_ps = pd.concat([m1, m2], axis=1)

if beta_ps.columns.duplicated().any():
    print("Duplicate sample columns:", beta_ps.columns.duplicated().sum())
    beta_ps = beta_ps.loc[:, ~beta_ps.columns.duplicated(keep="first")]

if beta_ps.index.duplicated().any():
    print("Duplicate probes:", beta_ps.index.duplicated().sum())
    beta_ps = beta_ps[~beta_ps.index.duplicated(keep="first")]

X_ext_full = beta_ps.T.astype("float32")
X_ext_full.index = X_ext_full.index.astype(str)
X_ext_full.index.name = "gsm_id"

print("External beta (samples × CpGs):", X_ext_full.shape)

gse_ext = GEOparse.get_GEO(geo=EXT_GSE, destdir=str(EXT_SUBDIR / "geo_soft"), how="brief")


def age_from_gsm(gsm) -> float:
    chars = gsm.metadata.get("characteristics_ch1", [])
    blob = " ".join(str(x) for x in chars)
    m = re.search(r"(?i)age\s*:\s*([0-9]+(?:\.[0-9]+)?)", blob)
    if m:
        return float(m.group(1))
    compact = re.sub(r"\s+", "", blob)
    m2 = re.search(r"(?i)age:([0-9]+(?:\.[0-9]+)?)", compact)
    if m2:
        return float(m2.group(1))
    return float("nan")


age_ext = pd.Series(
    {gid: age_from_gsm(gsm) for gid, gsm in gse_ext.gsms.items()},
    dtype=float,
).dropna()
age_ext.index = age_ext.index.astype(str)

gsm_ids = {str(k) for k in gse_ext.gsms.keys()}
frac_gsm = float(np.mean([i in gsm_ids for i in X_ext_full.index]))
if frac_gsm < 0.5:
    title_to_gsm = {}
    for gid, gsm in gse_ext.gsms.items():
        gid = str(gid)
        for t in gsm.metadata.get("title", []) or []:
            ts = str(t).strip()
            title_to_gsm[ts] = gid
            m = re.search(r"(?i)^X(\d+)\b", ts)
            if m:
                title_to_gsm["X" + m.group(1)] = gid
    X_ext_full.index = [
        i if i in gsm_ids else title_to_gsm.get(i, i) for i in X_ext_full.index
    ]
    if pd.Index(X_ext_full.index).duplicated().any():
        X_ext_full = X_ext_full[~pd.Index(X_ext_full.index).duplicated(keep="first")]

ext_cols = set(X_ext_full.columns.astype(str))
int_cols = set(X.columns.astype(str))
overlap_cpgs = [c for c in selected_cpgs if c in ext_cols and c in int_cols]
print("Model candidate CpGs:", len(selected_cpgs))
print("Overlapping CpGs (both cohorts):", len(overlap_cpgs))
missing_probes = [c for c in selected_cpgs if c not in ext_cols]
print("Training probes missing from external matrix:", len(missing_probes))

X_int = X[overlap_cpgs].copy()
common_ext = X_ext_full.index.intersection(age_ext.index)
X_ext_sub = X_ext_full.loc[common_ext, overlap_cpgs].dropna(how="any")
y_ext_a = age_ext.loc[X_ext_sub.index]

# Same leakage-safe Z as `04_external_validation.ipynb`: present probes scaled;
# absent probes → z=0.
mean_tr = scaler.mean_
scale_tr = scaler.scale_
scale_safe = np.where(scale_tr == 0, 1.0, scale_tr)
sub_z = X_ext_full.reindex(index=X_ext_sub.index)
Z_ext = np.zeros((len(X_ext_sub), len(selected_cpgs)), dtype=np.float64)
for j, cpg in enumerate(selected_cpgs):
    if cpg not in sub_z.columns:
        Z_ext[:, j] = 0.0
        continue
    col = sub_z[cpg].to_numpy(dtype=np.float64)
    Z_ext[:, j] = (col - mean_tr[j]) / scale_safe[j]

pred_ext = pd.Series(
    model.predict(Z_ext), index=X_ext_sub.index, name="predicted_age"
)

meta_int = pd.DataFrame(
    {
        "sample_id": X_int.index.astype(str),
        "cohort": "GSE40279",
        "chronological_age": y_age.loc[X_int.index].values,
        "predicted_age": pred_age.loc[X_int.index].values,
        "age_acceleration": (
            pred_age.loc[X_int.index] - y_age.loc[X_int.index]
        ).values,
    }
).set_index("sample_id")

meta_ext = pd.DataFrame(
    {
        "sample_id": X_ext_sub.index.astype(str),
        "cohort": "GSE87571",
        "chronological_age": y_ext_a.values,
        "predicted_age": pred_ext.loc[X_ext_sub.index].values,
        "age_acceleration": (
            pred_ext.loc[X_ext_sub.index] - y_ext_a
        ).values,
    }
).set_index("sample_id")

batch_meta = pd.concat([meta_int, meta_ext], axis=0)
batch_meta_path = RESULTS_DIR / "cohort_batch_metadata.csv"
batch_meta.to_csv(batch_meta_path, index_label="sample_id")
print("Combined metadata rows:", batch_meta.shape[0])
display(batch_meta.head())


### PCA on combined methylation (overlapping model CpGs)


In [ ]:
X_combined = pd.concat(
    [
        X_int,
        X_ext_full.loc[X_ext_sub.index, overlap_cpgs],
    ],
    axis=0,
).astype(np.float64)

meta_aligned = batch_meta.loc[X_combined.index]

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X_combined))

fig, ax = plt.subplots(figsize=(7, 6))
for lab, color in zip(["GSE40279", "GSE87571"], ["#1f77b4", "#ff7f0e"]):
    m = meta_aligned["cohort"] == lab
    ax.scatter(
        X_pca[m, 0],
        X_pca[m, 1],
        s=22,
        alpha=0.65,
        label=lab,
        c=color,
    )
ax.set_xlabel("PC1 ({:.1f}% var)".format(100 * pca.explained_variance_ratio_[0]))
ax.set_ylabel("PC2 ({:.1f}% var)".format(100 * pca.explained_variance_ratio_[1]))
ax.set_title("PCA of combined betas — colored by cohort")
ax.legend()
fig.tight_layout()
pca_cohort_path = RESULTS_DIR / "pca_by_cohort.png"
fig.savefig(pca_cohort_path, dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(7, 6))
sc = ax2.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=meta_aligned["chronological_age"].values,
    cmap="viridis",
    s=22,
    alpha=0.75,
)
fig2.colorbar(sc, ax=ax2, label="Chronological age")
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.set_title("PCA of combined betas — colored by chronological age")
fig2.tight_layout()
pca_age_path = RESULTS_DIR / "pca_by_age.png"
fig2.savefig(pca_age_path, dpi=150)
plt.show()

for pth in (pca_cohort_path, pca_age_path):
    shutil.copy2(pth, PUBLIC_DIR / pth.name)


### Optional UMAP


In [ ]:
try:
    import umap

    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
    X_um = reducer.fit_transform(StandardScaler().fit_transform(X_combined))
    figu, axu = plt.subplots(figsize=(7, 6))
    for lab, color in zip(["GSE40279", "GSE87571"], ["#1f77b4", "#ff7f0e"]):
        m = meta_aligned["cohort"] == lab
        axu.scatter(
            X_um[m, 0], X_um[m, 1], s=22, alpha=0.65, label=lab, c=color
        )
    axu.set_title("UMAP — cohort")
    axu.legend()
    figu.tight_layout()
    upath = RESULTS_DIR / "umap_by_cohort.png"
    figu.savefig(upath, dpi=150)
    plt.show()
    shutil.copy2(upath, PUBLIC_DIR / upath.name)
    figu2, axu2 = plt.subplots(figsize=(7, 6))
    sc_age = axu2.scatter(
        X_um[:, 0],
        X_um[:, 1],
        c=meta_aligned["chronological_age"].values,
        cmap="viridis",
        s=22,
        alpha=0.75,
    )
    figu2.colorbar(sc_age, ax=axu2, label="Age")
    axu2.set_title("UMAP — chronological age")
    figu2.tight_layout()
    upath2 = RESULTS_DIR / "umap_by_age.png"
    figu2.savefig(upath2, dpi=150)
    plt.show()
    shutil.copy2(upath2, PUBLIC_DIR / upath2.name)
except ImportError:
    print(
        "umap-learn not installed; skip UMAP. Optional: pip install umap-learn"
    )


### Age and predicted-age distributions by cohort


In [ ]:
figd, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4))
for lab, color in zip(["GSE40279", "GSE87571"], ["#1f77b4", "#ff7f0e"]):
    sub = meta_aligned[meta_aligned["cohort"] == lab]["chronological_age"]
    axa.hist(sub, bins=25, alpha=0.55, label=lab, color=color, density=True)
axa.set_xlabel("Chronological age")
axa.set_ylabel("Density")
axa.set_title("Chronological age by cohort")
axa.legend()
for lab, color in zip(["GSE40279", "GSE87571"], ["#1f77b4", "#ff7f0e"]):
    sub = meta_aligned[meta_aligned["cohort"] == lab]["predicted_age"]
    axb.hist(sub, bins=25, alpha=0.55, label=lab, color=color, density=True)
axb.set_xlabel("Predicted age (frozen clock)")
axb.set_ylabel("Density")
axb.set_title("Predicted age by cohort")
axb.legend()
figd.tight_layout()
dist_path = RESULTS_DIR / "cohort_age_prediction_distributions.png"
figd.savefig(dist_path, dpi=150)
plt.show()
shutil.copy2(dist_path, PUBLIC_DIR / dist_path.name)


### Cohort separability classifier

Train a simple model to predict **which study** a sample belongs to from **raw overlapping betas**. **ROC-AUC** near 1.0 means methylation features separate cohorts almost perfectly—consistent with strong batch/study-specific structure (not proof that external validation is "wrong," but a warning that signals may be partially study-specific).


In [ ]:
y_cohort = (meta_aligned["cohort"] == "GSE87571").astype(int).values
X_mat = X_combined.values
X_clf_tr, X_clf_te, y_clf_tr, y_clf_te = train_test_split(
    X_mat,
    y_cohort,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_cohort,
)

pipe_lr = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=5000,
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)
pipe_lr.fit(X_clf_tr, y_clf_tr)
proba_te = pipe_lr.predict_proba(X_clf_te)[:, 1]
pred_te = pipe_lr.predict(X_clf_te)
auc_lr = float(roc_auc_score(y_clf_te, proba_te))
acc_lr = float(accuracy_score(y_clf_te, pred_te))

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_depth=12,
)
rf.fit(X_clf_tr, y_clf_tr)
proba_rf = rf.predict_proba(X_clf_te)[:, 1]
pred_rf = rf.predict(X_clf_te)
auc_rf = float(roc_auc_score(y_clf_te, proba_rf))
acc_rf = float(accuracy_score(y_clf_te, pred_rf))

clf_metrics = pd.DataFrame(
    [
        {
            "model": "logistic_regression_scaled",
            "roc_auc_holdout": auc_lr,
            "accuracy_holdout": acc_lr,
            "n_features": X_mat.shape[1],
            "n_train": int(X_clf_tr.shape[0]),
            "n_test": int(X_clf_te.shape[0]),
        },
        {
            "model": "random_forest",
            "roc_auc_holdout": auc_rf,
            "accuracy_holdout": acc_rf,
            "n_features": X_mat.shape[1],
            "n_train": int(X_clf_tr.shape[0]),
            "n_test": int(X_clf_te.shape[0]),
        },
    ]
)
display(clf_metrics)

clf_path = RESULTS_DIR / "cohort_classifier_metrics.csv"
clf_metrics.to_csv(clf_path, index=False)
shutil.copy2(clf_path, PUBLIC_DIR / clf_path.name)


### Interpretation (batch vs biology)

- **Strong PCA/UMAP separation by cohort** together with **high cohort-classifier AUC** suggests that **study-specific methylation structure** is prominent relative to within-study age gradients projected into two dimensions. That does **not** prove the clock tracks only batch—it means naive cross-cohort visualization highlights technical divergence.
- **External validation** can still show reasonable age correlation if **age-aligned signal** overlaps technical axes—but disentangling the two requires **harmonization**, **cell adjustment**, or **prospective** study designs beyond this repository.
- This notebook **does not** apply **ComBat** or remove batch effects from predictions; it **flags** when cohort classification is trivially easy.


### External validation vs batch structure — synthesis

- **Biologically plausible signal:** If predicted ages track chronological ages **within** each cohort and **correlations** in notebook 04 are strong, part of the signal is consistent with shared aging biology.
- **Cohort-structure caveat:** High separability implies **external metrics** should be interpreted as **transport under technical shift**, not as a pure aging biomarker validation.
- **Limitations:** Without **full harmonization** (e.g., ComBat on betas, careful normalization parity, cell-type correction), residual batch effects likely remain; this diagnostic layer **does not replace** those pipelines.


## Conclusion template (fill after Run All)

1. **Immune dominance:** If |Pearson r| between acceleration and one or more fractions is large (especially neutrophil vs lymphoid contrasts), the clock’s residuals may be **partly compositional** — interpret biological acceleration cautiously.
2. **After adjustment:** If the optional model runs, compare **`cell_adjusted_model_metrics.csv`** to **`model_metrics.csv`**. Meaningful **MAE/R²** moves with shifted **`cg16867657`** coefficients indicate sensitivity to leukocyte covariates; stability of performance suggests **CpG signal beyond coarse composition**.
3. **Implications:** Persistent predictive accuracy after adding cell types supports **probe-level aging information** not fully explained by six proportions; the converse motivates **cell-aware** clocks or reference-free surrogate correction in EWAS-style analyses.